# 0장 실습 — 환경 준비와 첫 시뮬레이션

교재 0장을 옆에 놓고 이 노트북을 위에서부터 실행합니다.
셀이 전부 오류 없이 돌면 이번 학기 실습 환경이 준비된 것입니다.

마지막 빈칸 하나를 채우면 끝납니다. 20분 정도 걸립니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo

print("저장소:", ROOT)

## 1. 설치 확인 (교재 0.1)

`smartmob` 이 불러와지는지, 그리고 이 책이 기대하는 패키지가 깔려 있는지 봅니다.

In [ ]:
import matplotlib
import pandas as pd

import smartmob

banner("설치된 것")
print(f"smartmob     {smartmob.__version__}")
print(f"pandas       {pd.__version__}")
print(f"matplotlib   {matplotlib.__version__}")

한글이 깨지지 않게 폰트를 잡아 둡니다. 이 줄은 그림을 그리는 노트북마다 나옵니다.

In [ ]:
from smartmob.viz import use_korean_font

use_korean_font()

## 2. 데이터 확인 (교재 0.2)

실습 도시는 하남시입니다. 도로망과 수요가 저장소에 이미 들어 있습니다.

In [ ]:
from smartmob.data import data_path

for name in ["road_graph_nodes.parquet", "road_graph_edges.parquet", "demand.csv"]:
    p = data_path(f"hanam/{name}")
    print(f"{name:28s} {p.stat().st_size / 1e6:6.2f} MB")

도로망을 읽습니다. `modes=("drive",)` 는 자동차가 다닐 수 있는 도로만 남기라는 뜻입니다.
왜 이 인자가 필요한지는 2장에서 다룹니다.

In [ ]:
from smartmob.data import load_road_graph

G = load_road_graph("hanam", modes=("drive",))

banner("하남시 자동차 도로망")
expect("노드 수", G.n_nodes, 12_566)
expect("엣지 수", G.n_edges, 28_589)

두 값이 교재와 같으면 데이터가 제대로 들어온 것입니다.

## 3. 엔진에 연결하기 (교재 0.3)

시뮬레이션을 실제로 돌리는 것은 **DTUMOS** 라는 별도 프로그램입니다.
여기서는 HTTP로 부르기만 합니다.

서버에 못 붙으면 저장소에 녹화된 실행 결과를 대신 씁니다.
그래서 집에서도 이 노트북이 끝까지 돌아갑니다.

In [ ]:
from smartmob import Dtumos

dt = Dtumos()
print(dt.health())
print("사용 중인 모드:", dt.mode)

`status` 가 `fixture` 로 나오면 녹화본을 쓰는 중입니다.
실서버에 붙었다면 서버가 돌려준 상태가 그대로 나옵니다. 둘 다 정상입니다.

## 4. 첫 시뮬레이션 (교재 0.4)

하남시에서 저녁 6시부터 자정까지, 택시 80대로 1,000건의 호출을 처리합니다.
`1080` 은 자정부터의 분이고 18:00입니다. 이 책의 시간은 전부 이 단위입니다.

아래 인자는 녹화본과 똑같이 맞춰 두었습니다.
값을 바꾸면 실서버가 없을 때 `FixtureMissing` 오류가 납니다.
대수를 바꿔 가며 실험하는 것은 11장과 12장에서 직접 짠 루프로 합니다.

In [ ]:
REFERENCE = dict(
    city="hanam",
    mode="taxi",
    fleet_size=80,
    num_passengers=1000,
    time_start=1080,        # 18:00
    time_end=1440,          # 24:00
    dispatch_mode="optimization",
    matrix_mode="street_distance",
    vehicle_capacity=1,
    random_seed=42,
)

sim = dt.run_simulation(**REFERENCE)
sim.summary()

숫자를 하나씩 읽습니다.

- `service_rate` 가 1.0입니다. 990건의 호출이 전부 배차됐습니다.
- `avg_waiting_time_min` 이 약 4.1분입니다. 호출하고 차가 올 때까지 평균 4분 걸렸습니다.
- `utilization` 이 약 0.27입니다. 차량이 승객을 태우고 있던 시간이 전체의 27%뿐입니다.

마지막 값이 이 과목의 출발점입니다. 승객은 4분만 기다렸는데 차량의 4분의 3은 놀았습니다.
80대가 너무 많은 것인지, 40대로 줄이면 대기시간이 얼마나 늘어나는지가 다음 질문입니다.

## 5. 시간에 따라 무슨 일이 있었는가 (교재 0.5)

In [ ]:
sim.record.head()

In [ ]:
from smartmob.viz import plot_record

plot_record(sim.record);

운행 중 차량이 늘어난 만큼 대기 중 차량이 줄어듭니다. 둘을 더하면 대부분의 시간에 80이 됩니다.
마지막 30분에서 합이 80보다 작아지는 것은 근무 시간이 끝난 차량이 빠지기 때문입니다.

In [ ]:
on_duty = sim.record["empty_vehicle_cnt"] + sim.record["driving_vehicle_cnt"]
print("근무 중 차량 최대:", on_duty.max())
print("근무 중 차량 최소:", on_duty.min())

## 6. 빈칸

대기 승객이 가장 많았던 시각과 그때의 인원을 구합니다.
`sim.record` 에서 `waiting_passenger_cnt` 가 가장 큰 행을 찾으면 됩니다.

`minutes_to_hhmm` 이 분을 `HH:MM` 으로 바꿔 줍니다.

In [ ]:
from smartmob.data import minutes_to_hhmm

# 여기를 채웁니다. 힌트: idxmax() 로 가장 큰 행의 위치를 얻습니다.
peak_minute = None      # 대기 승객이 가장 많았던 시각 (분)
peak_waiting = None     # 그때의 대기 승객 수 (명)

banner("빈칸 확인")
todo("가장 붐빈 시각", peak_minute, fmt=minutes_to_hhmm)
todo("그때의 대기 승객", peak_waiting)

## 정리

- `load_road_graph("hanam")` 이 도로망을, `Dtumos()` 가 시뮬레이션 엔진을 담당합니다
- 엔진에 못 붙으면 `data/fixtures/` 의 녹화본으로 자동 전환됩니다
- `sim.summary()` 는 서비스율·평균 대기시간·차량 가동률을, `sim.record` 는 분 단위 시계열을 줍니다
- 1장 실습에서는 이 결과의 평균 뒤에 무엇이 가려져 있는지 봅니다